# 모듈 03: 조건부 흐름(Conditional Flow)

모듈 02에서 상태 설계와 부분 업데이트 메커니즘을 익혔습니다.
이번에는 상태 값에 따라 실행 경로를 동적으로 분기하는 방법을 배웁니다.

## 이 노트북에서 할 것
| 단계 | 내용 |
|------|------|
| 1 | 일반 엣지 vs 조건부 엣지 개념 비교 |
| 2 | 라우터 함수(Router Function) 작성 패턴 |
| 3 | add_conditional_edges() API로 분기 등록 |
| 4 | 2방향·3방향 분기 그래프 실습 |
| 5 | END를 반환하는 조건부 종료 패턴 |

예상 소요 시간: 약 30분  
전제 조건: 모듈 02 완료 (TypedDict 상태 설계, 부분 업데이트)

## 학습 목표

1. 라우터 함수가 상태를 읽고 다음 노드 이름(문자열)을 반환하는 규칙을 이해한다
2. `add_conditional_edges(출발_노드, 라우터, 매핑_딕셔너리)` 3개 인자의 역할을 설명할 수 있다
3. 조건부 엣지로 2방향·3방향 분기와 조건부 종료(`END` 반환) 패턴을 구현할 수 있다

## 전체 흐름도

```
┌──────────────────────────────────────────────────────┐
│                                                      │
│  [1] 일반 엣지 vs 조건부 엣지 개념 비교              │
│       ↓                                              │
│  [2] 라우터 함수 작성 패턴 (상태 → 노드 이름)        │
│       ↓                                              │
│  [3] add_conditional_edges() API 실습                │
│       ↓                                              │
│  [4] 2방향 분기 그래프 (합격/불합격)                 │
│       ↓                                              │
│  [5] 3방향 분기 그래프 (위험/주의/정상)              │
│       ↓                                              │
│  [6] END 반환: 조건부 종료 패턴                      │
│                                                      │
└──────────────────────────────────────────────────────┘
```

---

## 섹션 1: 일반 엣지 vs 조건부 엣지

### 일반 엣지: 항상 동일한 다음 노드로 이동

```
[check] ──────────────────→ [approve]   (항상)
```

### 조건부 엣지: 상태에 따라 다른 노드로 분기

```
[check] ──(score>=60)──→ [approve]
        ──(score<60)───→ [reject]
```

| 구분 | 메서드 | 동작 |
|------|--------|------|
| 일반 엣지 | `add_edge("A", "B")` | A 실행 후 항상 B로 이동 |
| 조건부 엣지 | `add_conditional_edges("A", router, {...})` | A 실행 후 router 결과에 따라 분기 |

In [ ]:
import os
import operator
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END

notebook_dir = os.path.dirname(os.path.abspath('__file__'))
langgraph_root = os.path.dirname(notebook_dir)
project_root = os.path.dirname(langgraph_root)
langchain_env = os.path.join(project_root, 'LangChain', '.env')
langgraph_env = os.path.join(langgraph_root, '.env')

if os.path.exists(langchain_env):
    load_dotenv(langchain_env)
    print('[OK] LangChain/.env 로드 완료')
elif os.path.exists(langgraph_env):
    load_dotenv(langgraph_env)
    print('[OK] LangGraph/.env 로드 완료')
else:
    print('[FAIL] .env 파일을 찾을 수 없습니다')

api_key = os.getenv('GOOGLE_API_KEY')
print(f'[OK] GOOGLE_API_KEY: {"설정됨" if api_key else "[FAIL] 미설정"}')

In [ ]:
# 일반 엣지: 항상 동일한 경로
class SimpleState(TypedDict):
    value: int
    result: str

def check(state: SimpleState) -> dict:
    print(f'  [check] value={state["value"]}')
    return {}

def process(state: SimpleState) -> dict:
    return {"result": f'처리 완료 (value={state["value"]})'}

normal_graph = StateGraph(SimpleState)
normal_graph.add_node("check", check)
normal_graph.add_node("process", process)
normal_graph.set_entry_point("check")
normal_graph.add_edge("check", "process")   # 항상 process로
normal_graph.add_edge("process", END)
normal_app = normal_graph.compile()

print('=== 일반 엣지: value=10 ===')
r1 = normal_app.invoke({"value": 10, "result": ""})
print(f'  결과: {r1["result"]}')

print('=== 일반 엣지: value=30 ===')
r2 = normal_app.invoke({"value": 30, "result": ""})
print(f'  결과: {r2["result"]}')

print()
print('[!] value가 달라도 항상 같은 경로를 탑니다')

---

## 섹션 2: 라우터 함수 패턴

### 라우터 함수 작성 규칙
- 입력: 현재 상태 딕셔너리
- 반환: 다음으로 이동할 노드 이름(문자열) 또는 `END`

```python
def router(state: MyState) -> str:
    if state["score"] >= 60:
        return "approve"   # 노드 이름
    return "reject"        # 노드 이름
```

라우터가 반환한 문자열이 매핑 딕셔너리의 키와 매칭됩니다:

```python
graph.add_conditional_edges(
    "check",                           # 분기 출발 노드
    router,                            # 라우터 함수
    {"approve": "approve_node",        # 반환값 → 이동할 노드
     "reject":  "reject_node"}
)
```

In [ ]:
class ScoreState(TypedDict):
    score: int
    verdict: str

def score_router(state: ScoreState) -> str:
    """점수에 따라 합격/불합격 분기"""
    if state["score"] >= 60:
        return "pass"
    return "fail"

# 그래프 없이 라우터 함수 단독 테스트
test_cases = [
    {"score": 85, "verdict": ""},
    {"score": 45, "verdict": ""},
    {"score": 60, "verdict": ""},
]

print('라우터 함수 단독 테스트:')
for case in test_cases:
    result = score_router(case)
    print(f'  score={case["score"]} → "{result}"')

print()
print('[OK] 라우터 함수는 일반 파이썬 함수입니다')
print('[OK] 그래프와 분리하여 단독으로 테스트할 수 있습니다')

---

## 섹션 3: add_conditional_edges() 3개 인자

```python
graph.add_conditional_edges(
    "check",        # ① 분기 출발 노드 이름
    router,         # ② 라우터 함수 (상태 → 문자열 반환)
    {               # ③ 반환값 → 이동할 노드 매핑
        "pass":  "approve",
        "fail":  "reject",
    }
)
```

| 인자 | 타입 | 설명 |
|------|------|------|
| ① 출발 노드 | `str` | 이 노드 실행 후 분기가 일어남 |
| ② 라우터 함수 | `Callable` | 상태를 받아 문자열 반환 |
| ③ 매핑 딕셔너리 | `dict` | 라우터 반환값 → 다음 노드 이름 |

**주의**: 매핑 딕셔너리의 키는 라우터 함수가 반환할 수 있는 모든 값을 포함해야 합니다.

In [ ]:
def evaluate(state: ScoreState) -> dict:
    """점수를 평가하는 노드 (아직 verdict는 비어 있음)"""
    print(f'  [evaluate] score={state["score"]}')
    return {}

def approve(state: ScoreState) -> dict:
    return {"verdict": f'합격 (점수: {state["score"]})'}

def reject(state: ScoreState) -> dict:
    return {"verdict": f'불합격 (점수: {state["score"]})'}

score_graph = StateGraph(ScoreState)
score_graph.add_node("evaluate", evaluate)
score_graph.add_node("approve", approve)
score_graph.add_node("reject", reject)
score_graph.set_entry_point("evaluate")

score_graph.add_conditional_edges(
    "evaluate",
    score_router,
    {"pass": "approve", "fail": "reject"}
)
score_graph.add_edge("approve", END)
score_graph.add_edge("reject", END)
score_app = score_graph.compile()

print('=== 2방향 분기: 합격/불합격 ===')
for score in [85, 60, 45]:
    r = score_app.invoke({"score": score, "verdict": ""})
    print(f'  score={score} → {r["verdict"]}')

---

## 섹션 4: 3방향 분기 패턴

3방향(또는 N방향) 분기도 동일한 패턴으로 구현합니다:

```
               ┌──(high)──→ [safe]
[check_level] ─┤──(medium)→ [warning]
               └──(low)───→ [danger]
```

라우터 함수가 3가지 이상의 값을 반환하면 됩니다:

```python
def level_router(state) -> str:
    value = state["value"]
    if value >= 80:
        return "high"
    elif value >= 40:
        return "medium"
    return "low"
```

매핑 딕셔너리에 모든 반환값을 키로 등록합니다:

```python
graph.add_conditional_edges(
    "check_level", level_router,
    {"high": "safe", "medium": "warning", "low": "danger"}
)
```

In [ ]:
class LevelState(TypedDict):
    value: int
    status: str

def level_router(state: LevelState) -> str:
    v = state["value"]
    if v >= 80:
        return "high"
    elif v >= 40:
        return "medium"
    return "low"

def check_level(state: LevelState) -> dict:
    print(f'  [check_level] value={state["value"]}')
    return {}

def safe(state: LevelState) -> dict:
    return {"status": f'정상 (value={state["value"]})'}

def warning(state: LevelState) -> dict:
    return {"status": f'주의 (value={state["value"]})'}

def danger(state: LevelState) -> dict:
    return {"status": f'위험 (value={state["value"]})'}

level_graph = StateGraph(LevelState)
level_graph.add_node("check_level", check_level)
level_graph.add_node("safe", safe)
level_graph.add_node("warning", warning)
level_graph.add_node("danger", danger)
level_graph.set_entry_point("check_level")

level_graph.add_conditional_edges(
    "check_level", level_router,
    {"high": "safe", "medium": "warning", "low": "danger"}
)
for node in ["safe", "warning", "danger"]:
    level_graph.add_edge(node, END)
level_app = level_graph.compile()

print('=== 3방향 분기: 위험/주의/정상 ===')
for val in [90, 55, 20]:
    r = level_app.invoke({"value": val, "status": ""})
    print(f'  value={val} → {r["status"]}')

---

## 섹션 5: END를 반환하는 조건부 종료 패턴

라우터 함수에서 `END`를 직접 반환하면 그래프를 즉시 종료할 수 있습니다:

```python
from langgraph.graph import END

def router(state) -> str:
    if state["done"]:
        return END        # 즉시 종료
    return "continue"     # 계속 진행
```

매핑 딕셔너리에 `END`를 값으로 지정합니다:

```python
graph.add_conditional_edges(
    "process",
    router,
    {"continue": "process", END: END}  # 루프백 또는 종료
)
```

이 패턴은 모듈 05의 에이전트 루프에서 핵심적으로 사용됩니다.

In [ ]:
class CountState(TypedDict):
    count: int
    log: Annotated[list, operator.add]

def count_router(state: CountState) -> str:
    if state["count"] >= 3:
        return "done"
    return "continue"

def increment(state: CountState) -> dict:
    new_count = state["count"] + 1
    return {"count": new_count, "log": [f'카운트: {new_count}']}

count_graph = StateGraph(CountState)
count_graph.add_node("increment", increment)
count_graph.set_entry_point("increment")
count_graph.add_conditional_edges(
    "increment",
    count_router,
    {"continue": "increment", "done": END}
)
count_app = count_graph.compile()

result = count_app.invoke({"count": 0, "log": []})
print('=== 조건부 종료 (count >= 3이 되면 정지) ===')
print(f'최종 count: {result["count"]}')
print(f'실행 기록: {result["log"]}')
print('[OK] count가 3에 도달하자 그래프가 자동 종료됩니다')

---

## 섹션 6: 그래프 구조 시각화

`get_graph().print_ascii()`로 세 그래프의 분기 구조를 확인합니다.  
조건부 엣지가 있으면 ASCII 출력에서 여러 갈래의 화살표를 볼 수 있습니다.

In [ ]:
print('=== 2방향 분기 그래프 ===')
score_app.get_graph().print_ascii()

print()
print('=== 3방향 분기 그래프 ===')
level_app.get_graph().print_ascii()

print()
print('=== 조건부 종료 그래프 (루프) ===')
count_app.get_graph().print_ascii()

---

## 마무리

### 핵심 패턴 요약

```python
from typing import TypedDict
from langgraph.graph import StateGraph, END

class State(TypedDict):
    score: int
    verdict: str

# 라우터 함수: 상태 → 노드 이름(str) 또는 END
def router(state: State) -> str:
    return "approve" if state["score"] >= 60 else "reject"

# 노드 함수들
def check(state: State) -> dict:
    return {}

def approve(state: State) -> dict:
    return {"verdict": "합격"}

def reject(state: State) -> dict:
    return {"verdict": "불합격"}

# 그래프 조립
graph = StateGraph(State)
graph.add_node("check", check)
graph.add_node("approve", approve)
graph.add_node("reject", reject)
graph.set_entry_point("check")

# 조건부 엣지 등록
graph.add_conditional_edges("check", router, {"approve": "approve", "reject": "reject"})
graph.add_edge("approve", END)
graph.add_edge("reject", END)

app = graph.compile()
result = app.invoke({"score": 75, "verdict": ""})
```

### 핵심 API 레퍼런스

| API | 설명 |
|-----|------|
| `add_conditional_edges(node, fn, map)` | 조건부 엣지 등록 |
| `router(state) -> str` | 라우터 함수 시그니처 |
| `return END` | 라우터에서 그래프 즉시 종료 |
| `{"key": "node_name"}` | 반환값 → 노드 이름 매핑 |
| `{"key": END}` | 반환값 → 종료 매핑 |

---

## 다음 모듈 예고: 모듈 04 - 대화형 챗봇

- `MessagesState`: 메시지 누적을 내장한 미리 정의된 상태 타입
- `add_messages` 리듀서: HumanMessage / AIMessage를 순서대로 쌓기
- LLM 호출 노드 + 멀티턴 대화 구현

코드 미리보기:

```python
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage

class ChatState(MessagesState):
    pass

def chatbot(state: ChatState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}
```

---

### 자기 점검 체크리스트

- [ ] 라우터 함수가 반환하는 값의 타입과 의미를 설명할 수 있나요?
- [ ] `add_conditional_edges()`의 세 인자를 순서대로 말할 수 있나요?
- [ ] 라우터에서 `END`를 반환하면 어떻게 되나요?